In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install trl
!pip install torchinfo
!pip install --upgrade "torchao>=0.16.0"
!pip install causal-conv1d>=1.2.0
!pip install mamba-ssm
import torch
import torch.nn as nn

from trl import SFTConfig, SFTTrainer

from torchinfo import summary
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from transformers import TrainingArguments, AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model_id = "state-spaces/mamba-130m-hf"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model1 = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map=device)

In [ ]:
model1.backbone.layers[0]

In [ ]:
summary(model1, input_size=(1, 16), dtypes=[torch.long])

In [ ]:
prompt = "Explain the theory of relativity"


inputs = tokenizer(prompt, return_tensors="pt").to(device)

output_tokens = model1.generate(
    **inputs,
    max_new_tokens=30,
    do_sample=True,
    temperature=0.7,
    top_k=50
)


result = tokenizer.decode(output_tokens[0], skip_special_tokens=True)

print(result)

In [ ]:
class SVDLinear(nn.Module):
    def __init__(self, in_features, out_features, rank, bias=True):

        super().__init__()
        self.v_layer = nn.Linear(in_features, rank, bias=False)
        self.u_layer = nn.Linear(rank, out_features, bias=bias)

    def forward(self, x):
        return self.u_layer(self.v_layer(x))
def apply_svd_to_linear(module, rank):
  for name, child in module.named_children():
    if isinstance(child, nn.Linear) and name in ["in_proj", "out_proj"]:

        in_features = child.in_features
        out_features = child.out_features


        current_rank = min(rank, in_features, out_features)


        W = child.weight.data.float()
        U, S, Vh = torch.linalg.svd(W, full_matrices=False)

        U_r = U[:, :current_rank]
        S_r = S[:current_rank]
        Vh_r = Vh[:current_rank, :]


        svd_module = SVDLinear(in_features, out_features, current_rank, bias=(child.bias is not None))


        svd_module.v_layer.weight.data = (torch.diag(S_r) @ Vh_r).to(child.weight.dtype)
        svd_module.u_layer.weight.data = U_r.to(child.weight.dtype)

        if child.bias is not None:
            svd_module.u_layer.bias.data = child.bias.data.clone()


        setattr(module, name, svd_module)
    else:

        apply_svd_to_linear(child, rank)

apply_svd_to_linear(model1, rank=128)

In [ ]:
summary(model1, input_size=(1, 16), dtypes=[torch.long])

In [ ]:
prompt = "Explain the theory of relativity"


inputs = tokenizer(prompt, return_tensors="pt").to(device)

output_tokens = model1.generate(
    **inputs,
    max_new_tokens=30,
    do_sample=True,
    temperature=0.7,
    top_k=50
)


result = tokenizer.decode(output_tokens[0], skip_special_tokens=True)

print(result)

In [ ]:

print("Loading Wikitext Dataset...")

dataset = load_dataset("Salesforce/wikitext", "wikitext-2-v1", split="train")


dataset = dataset.filter(lambda x: len(x['text'].strip()) > 20)

print(f"Dataset ready!:\n{dataset[10]['text']}")



In [ ]:
###### POS tagging transformation of the dataset

!python -m spacy download en_core_web_sm
import spacy


nlp = spacy.load("en_core_web_sm")


def apply_pos_tags(example):

    doc = nlp(example['text'])
    tagged_words = []

    for token in doc:

        if token.text.strip():
            tagged_words.append(f"[{token.pos_}] {token.text}")

    return {"text": " ".join(tagged_words)}




tagged_dataset = dataset.map(apply_pos_tags, num_proc=4)

print("\nTagged dataset:")
print("-" * 50)
print(tagged_dataset[10]['text'][:500])
print("-" * 50)

In [ ]:
split_dataset =tagged_dataset.train_test_split(test_size=0.05, seed=42)

### No tagging
#split_dataset =dataset.train_test_split(test_size=0.05, seed=42)
####

train_data = split_dataset["train"]
val_data = split_dataset["test"]

print(f"Train size: {len(train_data)} | Validation size: {len(val_data)}")

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["u_layer", "v_layer"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    use_rslora=True,
    init_lora_weights="gaussian",
    modules_to_save=["lm_head"],
)

peft_model = get_peft_model(model1, lora_config)

peft_model.print_trainable_parameters()

In [ ]:
import os


current_rank = 128

sft_config = SFTConfig(
    output_dir=f"/content/drive/MyDrive/mamba-checkpoints-rank-{current_rank}-POS",
    run_name=f"mamba-wiki-text-rank-{current_rank}-POS",
    save_strategy="steps",
    save_steps=50,
    save_total_limit=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    max_steps=100,
    fp16=True,
    optim="adamw_torch",
    dataset_text_field="text",
    max_length=256,

)


trainer = SFTTrainer(
    model=peft_model,
    train_dataset=train_data,
    eval_dataset=val_data,
    args=sft_config,
)

for param in peft_model.parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)

trainer.train()

In [ ]:


save_path = f"/content/drive/MyDrive/Project_Generative_Weights/mamba-wiki-text-final-rank-{current_rank}-POS"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print(f"Model saved on: {save_path}")

In [ ]:
import matplotlib.pyplot as plt


history = trainer.state.log_history

train_steps = []
train_losses = []
eval_steps = []
eval_losses = []


for entry in history:

    if "loss" in entry and "step" in entry:
        train_steps.append(entry["step"])
        train_losses.append(entry["loss"])

    elif "eval_loss" in entry and "step" in entry:
        eval_steps.append(entry["step"])
        eval_losses.append(entry["eval_loss"])


plt.figure(figsize=(10, 6))

if train_losses:
    plt.plot(train_steps, train_losses, label="Training Loss", color="blue", marker="o")


if eval_losses:
    plt.plot(eval_steps, eval_losses, label="Validation Loss", color="red", marker="x", linestyle="--")


plt.xlabel("Steps", fontsize=12)
plt.ylabel("Loss", fontsize=12)
plt.title("Training vs Validation Loss", fontsize=14, fontweight="bold")
plt.legend()
plt.grid(True, linestyle=":", alpha=0.7)


plt.show()

In [ ]:
#### No tagging prompt
peft_model.eval()


art_prompt = "Leonardo da Vinci was one of the greatest painters of the Renaissance. He is most famous for"


inputs = tokenizer(art_prompt, return_tensors="pt").to(device)


with torch.no_grad():
    outputs = peft_model.generate(
        **inputs,
        max_new_tokens=50,
        temperature=0.7,
        do_sample=True,
        top_k=50,
        eos_token_id=tokenizer.eos_token_id
    )


result = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(result)

In [ ]:
import math

print("Evaluation...")


eval_results = trainer.evaluate()


eval_loss = eval_results["eval_loss"]


try:
    perplexity = math.exp(eval_loss)
except OverflowError:

    perplexity = float("inf")

print("\n" + "="*40)
print(f"Validation Loss: {eval_loss:.4f}")
print(f"Perplexity (PPL): {perplexity:.4f}")
print("="*40)

In [ ]:
import pandas as pd
import os
import math



run_name = sft_config.run_name

trainable_params, all_params = peft_model.get_nb_trainable_parameters()
original_params = 167750400
compression_percent = round((1 - (all_params / original_params)) * 100, 2)


history = trainer.state.log_history
eval_losses = [entry["eval_loss"] for entry in history if "eval_loss" in entry]
final_eval_loss = eval_losses[-1] if eval_losses else None


if final_eval_loss is not None:
    perplexity = math.exp(final_eval_loss)
else:
    perplexity = "N/A"


new_data = pd.DataFrame([{
    "Run ID": run_name,
    "SVD Rank": current_rank,
    "Total Params": all_params,
    "Trainable (LoRA)": trainable_params,
    "Compression %": f"{compression_percent}%",
    "Final Val Loss": round(final_eval_loss, 4) if final_eval_loss else "N/A",
    "Perplexity": round(perplexity, 2) if isinstance(perplexity, float) else perplexity
}])


csv_path = "/content/drive/MyDrive/mamba_svd_master_log.csv"

if os.path.exists(csv_path):

    new_data.to_csv(csv_path, mode='a', header=False, index=False)
    print(f"Rank {current_rank} | Perplexity: {new_data['Perplexity'][0]}")
else:

    new_data.to_csv(csv_path, index=False)
    print(f"New log: Rank {current_rank} | Perplexity: {new_data['Perplexity'][0]}")

In [ ]:
### Tagged prompt

import spacy
import torch


nlp = spacy.load("en_core_web_sm")
prompt_text = "Leonardo da Vinci was one of the greatest painters of the Renaissance. He is most famous for"


doc = nlp(prompt_text)
tagged_prompt = " ".join([f"[{token.pos_}] {token.text}" for token in doc if token.text.strip()])


print(tagged_prompt)
print("-" * 50)


peft_model.eval()


inputs = tokenizer(tagged_prompt, return_tensors="pt").to(device)


with torch.no_grad():
    output_tokens = peft_model.generate(
        **inputs,
        max_new_tokens=40,
        do_sample=True,
        temperature=0.7,
        top_k=50,
        repetition_penalty=1.2,
        eos_token_id=tokenizer.eos_token_id
    )


result = tokenizer.decode(output_tokens[0], skip_special_tokens=True)

print("\nFinal result:")
print(result)

In [ ]:
#from google.colab import runtime
#print("Η εκπαίδευση τελείωσε! Τερματισμός του session για εξοικονόμηση credits...")
#runtime.unassign()